# 05 ? Platform Segmentation: Isolating Mobile Web Friction
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Isolate the exact point in the customer journey where Mobile Web underperforms native apps and desktop.

---
### Hypotheses to Test:
- Is Mobile Web underperforming at search discovery?
- Is Mobile Web underperforming at PDP consideration?
- Is Mobile Web underperforming specifically at checkout payment/auth?


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
from exploratory_analysis import get_db_connection, two_proportion_z_test

con = get_db_connection()


## 1. Full Platform Funnel Progression Table


In [ ]:
q_plat_funnel = '''
SELECT 
    s.platform,
    COUNT(DISTINCT s.session_id) AS total_sessions,
    ROUND(100.0 * COUNT(DISTINCT pv.session_id) / COUNT(DISTINCT s.session_id), 2) AS s_to_pdp_pct,
    ROUND(100.0 * COUNT(DISTINCT ce.session_id) / COUNT(DISTINCT pv.session_id), 2) AS pdp_to_cart_pct,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT ce.session_id), 2) AS cart_to_order_pct,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT s.session_id), 2) AS session_to_order_pct
FROM sessions s
LEFT JOIN product_views pv ON s.session_id = pv.session_id
LEFT JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
GROUP BY s.platform
ORDER BY session_to_order_pct DESC
'''
df_pf = con.execute(q_plat_funnel).df()
print("Cross-Platform Funnel Benchmarks:")
print(df_pf.to_string())

# Statistical Test: iOS vs Mobile Web Cart->Order
q_raw = '''
SELECT 
    s.platform,
    COUNT(DISTINCT ce.session_id) AS carts,
    COUNT(DISTINCT o.session_id) AS orders
FROM sessions s
JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
WHERE s.platform IN ('iOS', 'Mobile Web')
GROUP BY s.platform
'''
df_raw = con.execute(q_raw).df()
ios_o, ios_c = df_raw.loc[df_raw['platform']=='iOS', ['orders', 'carts']].values[0]
mw_o, mw_c = df_raw.loc[df_raw['platform']=='Mobile Web', ['orders', 'carts']].values[0]

diff, z, p, ci = two_proportion_z_test(ios_o, ios_c, mw_o, mw_c)
print(f"\niOS vs Mobile Web Checkout Conversion Z-Test: Diff = +{diff*100:.2f} pp, Z = {z:.2f}, p-value = {p:.2e}")


## 2. Cross-Tabulation: Mobile Web x User Type & Loyalty Tier


In [ ]:
q_mw_user = '''
SELECT 
    s.platform,
    u.user_type,
    COUNT(DISTINCT ce.session_id) AS carts,
    COUNT(DISTINCT o.session_id) AS orders,
    ROUND(100.0 * COUNT(DISTINCT o.session_id) / COUNT(DISTINCT ce.session_id), 2) AS cart_to_order_pct
FROM sessions s
JOIN users u ON s.user_id = u.user_id
JOIN cart_events ce ON s.session_id = ce.session_id
LEFT JOIN orders o ON s.session_id = o.session_id
GROUP BY 1, 2
ORDER BY 1, 2
'''
print("Platform x User Type Checkout Performance:")
print(con.execute(q_mw_user).df().to_string())


## 3. Visual Evidence


In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/09_platform_funnel_comparison.png'))
display(Image(filename='../reports/figures/10_platform_x_shipping_status.png'))
